# Nestlé WISER DOM — complete experiment notebook

This notebook implements every experiment requested in the challenge and the solver-comparison evidence needed to interpret them. It starts with a strict readability gate over all ten supplied files. It never writes raw operational tables; only aggregate metrics are saved.

The `quantum_seed_noise` study perturbs a **local simulated QUBO**. It measures seed/coefficient robustness and is not evidence about physical QPU noise or quantum advantage.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from domopt.experiments import run_challenge_experiments, write_experiment_results
from domopt.poc import (
    PocConfig,
    audit_poc_bundle,
    audit_poc_outputs,
    load_poc_problem,
    prune_pareto_candidates,
)

BUNDLE_DIR = Path(os.environ.get("NESTLE_BUNDLE_DIR", "../upload"))
PROFILE = os.environ.get("NESTLE_EXPERIMENT_PROFILE", "full")
OUTPUT_PATH = Path("runs/challenge-study/aggregate_results.csv")
print({"bundle_dir": str(BUNDLE_DIR), "profile": PROFILE, "output": str(OUTPUT_PATH)})

## 1. Stop-on-error readability gate

This cell parses every CSV, opens the XLSX, validates the DOCX OOXML package, and extracts all PDF pages. Any unreadable file raises immediately and the notebook stops.

In [ ]:
file_audit = audit_poc_bundle(BUNDLE_DIR)
assert file_audit["readable"].all()
file_audit[["role", "filename", "rows", "columns", "readable"]]

## 2. Construct and audit the real POC model

The adapter converts planning units to cases, builds load-cohesive candidates, protects five future days of ATP, applies observed dock headroom, and reproduces the thresholded penalty equation. Throughput utilization is **not** treated as a hard maximum unless an explicit headroom scenario is requested.

In [ ]:
problem_unpruned = load_poc_problem(
    BUNDLE_DIR,
    config=PocConfig(pareto_prune=False),
    strict_bundle_audit=False,
)
provided_output_audit = audit_poc_outputs(BUNDLE_DIR, problem_unpruned)
provided_output_audit

In [ ]:
problem_pruned = prune_pareto_candidates(problem_unpruned)
pd.DataFrame(
    [
        {
            "variant": "unpruned",
            "orders": len(problem_unpruned.orders),
            "order_lines": len(problem_unpruned.order_lines),
            "candidates": len(problem_unpruned.candidates),
            "assignment_groups": problem_unpruned.orders["assignment_group"].nunique(),
        },
        {
            "variant": "pareto_pruned",
            "orders": len(problem_pruned.orders),
            "order_lines": len(problem_pruned.order_lines),
            "candidates": len(problem_pruned.candidates),
            "assignment_groups": problem_pruned.orders["assignment_group"].nunique(),
        },
    ]
)

## 3. Run the complete study

The full profile runs: solver comparison, size scaling, penalty sensitivity, candidate-count sensitivity, inventory shocks, simulator seed/noise robustness, Pareto-pruning ablation, conflict-vs-random batching, sampler ablation, and an independently generated coordination control. Set `NESTLE_EXPERIMENT_PROFILE=smoke` only for a fast engineering check.

In [ ]:
results = run_challenge_experiments(problem_unpruned, profile=PROFILE)
assert results["feasible"].fillna(False).all(), results.loc[~results["feasible"].fillna(False)]
write_experiment_results(results, OUTPUT_PATH)
print(f"wrote {len(results)} aggregate rows to {OUTPUT_PATH}")

## 4. Common solver comparison

All methods use the same canonical objective and independent feasibility validator. The exact MILP's gap is a proof certificate when it is zero. Hybrid improvement is always measured from its feasible incumbent.

In [ ]:
core = results.loc[results["experiment"] == "solver_comparison"]
core[[
    "method", "feasible", "objective_value", "case_fill_rate",
    "penalty_cost", "shipping_cost", "runtime_seconds",
    "optimality_gap", "hybrid_improvement"
]].sort_values("objective_value", ascending=False)

In [ ]:
ax = core.sort_values("objective_value").plot.barh(
    x="method", y="objective_value", legend=False, color="#1f77b4"
)
ax.set_title("Common-objective solver comparison")
ax.set_xlabel("objective (source currency)")
plt.tight_layout()

## 5. Size scaling

In [ ]:
scaling = results.loc[results["experiment"] == "size_scaling"].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for method, group in scaling.groupby("method"):
    group = group.sort_values("order_count")
    axes[0].plot(group["order_count"], group["objective_value"], marker="o", label=method)
    axes[1].plot(group["order_count"], group["runtime_seconds"], marker="o", label=method)
axes[0].set(title="Objective scaling", xlabel="actual orders", ylabel="objective")
axes[1].set(title="Runtime scaling", xlabel="actual orders", ylabel="seconds")
axes[0].legend(); axes[1].legend(); plt.tight_layout()

## 6. Penalty-weight, candidate-count, and inventory-shock sensitivity

In [ ]:
sensitivity = results.loc[results["experiment"].isin([
    "penalty_weight_sensitivity",
    "candidate_count_sensitivity",
    "inventory_shock",
])][[
    "experiment", "level", "method", "feasible", "objective_value",
    "case_fill_rate", "reassigned_orders", "runtime_seconds"
]]
sensitivity

## 7. Simulator seed and coefficient-noise robustness

These rows describe the local simulated QUBO only. The exact recourse/validator safety layer remains unchanged.

In [ ]:
noise = results.loc[results["experiment"] == "quantum_seed_noise"].copy()
noise[[
    "level", "objective_value", "hybrid_improvement",
    "accepted_moves", "maximum_qubo_variables", "runtime_seconds"
]].sort_values("level")

## 8. Pareto pruning and batching ablations

In [ ]:
ablations = results.loc[results["experiment"].isin([
    "pareto_pruning_ablation",
    "batch_strategy_ablation",
    "sampler_ablation",
])][[
    "experiment", "level", "candidate_count", "objective_value",
    "hybrid_improvement", "runtime_seconds", "feasible"
]]
ablations

## 9. Independently generated coordination control

This added control is designed to expose greedy myopia. It may demonstrate a hybrid-search benefit, but it is synthetic and must not be presented as quantum or real-data advantage.

In [ ]:
results.loc[results["experiment"] == "synthetic_coordination_control", [
    "method", "objective_value", "case_fill_rate", "runtime_seconds",
    "optimality_gap", "hybrid_improvement", "feasible"
]].sort_values("objective_value", ascending=False)

## 10. Interpretation guardrails

A defensible conclusion requires all returned solutions to remain feasible, hybrid objective never to fall below its recorded incumbent, and claims to separate real POC evidence from synthetic controls. Pareto pruning and conflict batching are useful when they reduce runtime/QUBO width or improve candidate discovery without changing feasibility. A physical quantum-advantage claim requires approved QPU execution, hardware metadata, matched wall-clock budgets, embedding overhead, uncertainty intervals, and repeated statistical tests; this notebook intentionally makes no such claim.